# Clean & reshape `CENSUS_TRACT`

Transforms `PUBLIC.MMG.CENSUS_TRACT` (Feeding America "Map the Meal Gap" x 5-Yr ACS) into a typed, tidy table `PUBLIC.MMG.CENSUS_TRACT_TRANSFORMED`.

- Source metrics arrive as **display strings** (`16.6%`, `$54,877`, `3,968`); this notebook parses them to real numeric types, drops the empty *Food Bank 2* columns, renames *Food Bank 1* -> *FoodBank*, and decomposes `Geography` into *Tract Number / County / State Name*.
- **Run top-to-bottom.** The write (Step 1) is a single self-contained `CREATE OR REPLACE TABLE`, so it is idempotent and safe to re-run.
- **Requires** `CREATE TABLE` on `PUBLIC.MMG`. If you only have read access there, change the target in Step 1 to a schema you own.

## Step 0 - Inspect the source

Confirm the **exact** column identifiers and types below. The transform in Step 1 uses double-quoted names copied from the data's display header; if your table's real identifiers differ (e.g. `TOTAL_POPULATION_5_YR_ACS`), reconcile them before running Step 1 or it errors with `invalid identifier`. Also eyeball that the rate columns look like `16.6%` (0-100 scale, not already-fractional `0.166`) and income like `$54,877`.

In [ ]:
%%sql -r dataframe_1
DESCRIBE TABLE PUBLIC.bronze.CENSUS_TRACT;

In [ ]:
%%sql -r dataframe_2
SELECT * FROM PUBLIC.bronze.CENSUS_TRACT LIMIT 20;

## Step 1 - Build the transformed table

One self-contained `CREATE OR REPLACE TABLE ... AS` (no dependency on prior cells' session state).

Transformations applied:
- **Drop** `Food Bank 2 ID`, `Food Bank 2`.
- **Rename** `Food Bank 1 ID` -> `FoodBank ID`, `Food Bank 1` -> `FoodBank`.
- **Trim + empty->NULL** on every field.
- **Percents** (food insecurity, unemployment, poverty, percent black/hispanic, homeownership, disability): strip `%`, divide by 100, round to 4dp -> `16.6%` becomes `0.166`.
- **Currency** `Median Income`: strip `$` and `,` -> `NUMBER(18,2)`.
- **Counts** `Total Population`, `# of Food Insecure Persons Overall`: strip `,` -> `INTEGER`.
- **Identifiers** (`State FIPS`, `County FIPS`, `Tract ID`, `FoodBank ID`) kept as strings to preserve leading zeros.
- **Decompose `Geography`** (normalizing `;` -> `,` first) into `Tract Number` / `County` / `State Name`; original `Geography`, `County, State`, and `State` are retained.

*Dry-run before writing:* replace the `CREATE OR REPLACE TABLE ... AS` line with just the inner `SELECT` (add `LIMIT 50`) to eyeball output first.

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE DYNAMIC TABLE public.silver.census_tract
    TARGET_LAG = DOWNSTREAM
    WAREHOUSE = DEV01_TESTING
AS
WITH src AS (
  SELECT
    *,
    -- normalize the ; -> , delimiter inconsistency before splitting Geography
    REPLACE(NULLIF(TRIM("Geography"), ''), ';', ',') AS geo_norm
  FROM PUBLIC.bronze.CENSUS_TRACT
)
SELECT
  -- identifiers: trimmed strings, empty->null (no numeric cast: preserves leading zeros)
  NULLIF(TRIM("State FIPS"), '')                                     AS "State FIPS",
  NULLIF(TRIM("County FIPS"), '')                                    AS "County FIPS",
  NULLIF(TRIM("Tract ID"), '')                                       AS "Tract ID",
  NULLIF(TRIM("Geography"), '')                                      AS "Geography",
  NULLIF(TRIM("County, State"), '')                                  AS "County, State",
  NULLIF(TRIM("State"), '')                                          AS "State",
  TRY_CAST(NULLIF(TRIM("Year"), '') AS INTEGER)                      AS "Year",

  -- food bank (renamed; Food Bank 2 columns dropped by omission)
  NULLIF(TRIM("Food Bank 1 ID"), '')                                 AS "FoodBank ID",
  NULLIF(TRIM("Food Bank 1"), '')                                    AS "FoodBank",

  -- counts: strip commas -> integer
  TRY_CAST(REPLACE(NULLIF(TRIM("Total Population (5 Yr ACS)"), ''), ',', '') AS INTEGER)
                                                                     AS "Total Population (5 Yr ACS)",
  TRY_CAST(REPLACE(NULLIF(TRIM("# of Food Insecure Persons Overall"), ''), ',', '') AS INTEGER)
                                                                     AS "# of Food Insecure Persons Overall",

  -- percents: strip %, /100, round to 4dp
  ROUND(TRY_CAST(REPLACE(NULLIF(TRIM("Overall Food Insecurity Rate"), ''), '%', '') AS FLOAT) / 100, 4)
                                                                     AS "Overall Food Insecurity Rate",
  ROUND(TRY_CAST(REPLACE(NULLIF(TRIM("Unemployment Rate (5 Yr ACS)"), ''), '%', '') AS FLOAT) / 100, 4)
                                                                     AS "Unemployment Rate (5 Yr ACS)",
  ROUND(TRY_CAST(REPLACE(NULLIF(TRIM("Poverty Rate (5 Yr ACS)"), ''), '%', '') AS FLOAT) / 100, 4)
                                                                     AS "Poverty Rate (5 Yr ACS)",
  ROUND(TRY_CAST(REPLACE(NULLIF(TRIM("Percent Black (all ethnicities) (5 Yr ACS)"), ''), '%', '') AS FLOAT) / 100, 4)
                                                                     AS "Percent Black (all ethnicities) (5 Yr ACS)",
  ROUND(TRY_CAST(REPLACE(NULLIF(TRIM("Percent Hispanic (any race) (5 Yr ACS)"), ''), '%', '') AS FLOAT) / 100, 4)
                                                                     AS "Percent Hispanic (any race) (5 Yr ACS)",
  ROUND(TRY_CAST(REPLACE(NULLIF(TRIM("Homeownership Rate (5 Yr ACS)"), ''), '%', '') AS FLOAT) / 100, 4)
                                                                     AS "Homeownership Rate (5 Yr ACS)",
  ROUND(TRY_CAST(REPLACE(NULLIF(TRIM("Disability Rate (5 Yr ACS)"), ''), '%', '') AS FLOAT) / 100, 4)
                                                                     AS "Disability Rate (5 Yr ACS)",

  -- currency: strip $ and , -> number
  TRY_CAST(REPLACE(REPLACE(NULLIF(TRIM("Median Income (5 Yr ACS)"), ''), '$', ''), ',', '') AS NUMBER(18,2))
                                                                     AS "Median Income (5 Yr ACS)",

  -- Geography decomposition (delimiter already normalized to ',')
  NULLIF(TRIM(REPLACE(SPLIT_PART(geo_norm, ',', 1), 'Census Tract', '')), '') AS "Tract Number",
  NULLIF(TRIM(SPLIT_PART(geo_norm, ',', 2)), '')                     AS "County",
  NULLIF(TRIM(SPLIT_PART(geo_norm, ',', 3)), '')                     AS "State Name"
FROM src;

## Step 2 - Preview the result

Confirm the percent columns are now decimals <= 1, `Median Income` is numeric, and `Tract Number` / `County` / `State Name` are populated for both comma- and semicolon-delimited `Geography` rows.

In [ ]:
%%sql -r dataframe_4
SELECT * FROM PUBLIC.silver.CENSUS_TRACT LIMIT 50;

## Step 3 - Validate

One row of sanity checks:
- `source_rows` = `transformed_rows` (no rows lost).
- `max_rate_should_be_le_1` <= 1 across all seven percent columns; if > 1 the /100 assumption was wrong for some column.
- every `*_parse_fails` = 0 (values that were non-empty in the source but did not parse -- e.g. stray tokens like `N/A`, `(X)`, `-`). Parse-failure checks run against the SOURCE table because the transformed table no longer holds the raw strings.

In [ ]:
%%sql -r dataframe_5
SELECT
  (SELECT COUNT(*) FROM PUBLIC.bronze.CENSUS_TRACT)             AS source_rows,
  (SELECT COUNT(*) FROM PUBLIC.silver.CENSUS_TRACT) AS transformed_rows,
  (SELECT GREATEST(
      MAX("Overall Food Insecurity Rate"),
      MAX("Unemployment Rate (5 Yr ACS)"),
      MAX("Poverty Rate (5 Yr ACS)"),
      MAX("Percent Black (all ethnicities) (5 Yr ACS)"),
      MAX("Percent Hispanic (any race) (5 Yr ACS)"),
      MAX("Homeownership Rate (5 Yr ACS)"),
      MAX("Disability Rate (5 Yr ACS)"))
   FROM PUBLIC.silver.CENSUS_TRACT_TRANSFORMED)                 AS max_rate_should_be_le_1,

  -- parse-failure counts vs SOURCE (raw non-empty AND TRY_CAST -> NULL)
  (SELECT COUNT_IF(NULLIF(TRIM("Total Population (5 Yr ACS)"), '') IS NOT NULL
       AND TRY_CAST(REPLACE(NULLIF(TRIM("Total Population (5 Yr ACS)"), ''), ',', '') AS INTEGER) IS NULL)
   FROM PUBLIC.bronze.CENSUS_TRACT)                             AS total_population_parse_fails,
  (SELECT COUNT_IF(NULLIF(TRIM("# of Food Insecure Persons Overall"), '') IS NOT NULL
       AND TRY_CAST(REPLACE(NULLIF(TRIM("# of Food Insecure Persons Overall"), ''), ',', '') AS INTEGER) IS NULL)
   FROM PUBLIC.bronze.CENSUS_TRACT)                             AS food_insecure_persons_parse_fails,
  (SELECT COUNT_IF(NULLIF(TRIM("Median Income (5 Yr ACS)"), '') IS NOT NULL
       AND TRY_CAST(REPLACE(REPLACE(NULLIF(TRIM("Median Income (5 Yr ACS)"), ''), '$', ''), ',', '') AS NUMBER(18,2)) IS NULL)
   FROM PUBLIC.bronze.CENSUS_TRACT)                             AS median_income_parse_fails,
  (SELECT COUNT_IF(NULLIF(TRIM("Overall Food Insecurity Rate"), '') IS NOT NULL
       AND TRY_CAST(REPLACE(NULLIF(TRIM("Overall Food Insecurity Rate"), ''), '%', '') AS FLOAT) IS NULL)
   FROM PUBLIC.bronze.CENSUS_TRACT)                             AS food_insecurity_rate_parse_fails,
  (SELECT COUNT_IF(NULLIF(TRIM("Unemployment Rate (5 Yr ACS)"), '') IS NOT NULL
       AND TRY_CAST(REPLACE(NULLIF(TRIM("Unemployment Rate (5 Yr ACS)"), ''), '%', '') AS FLOAT) IS NULL)
   FROM PUBLIC.bronze.CENSUS_TRACT)                             AS unemployment_rate_parse_fails,
  (SELECT COUNT_IF(NULLIF(TRIM("Poverty Rate (5 Yr ACS)"), '') IS NOT NULL
       AND TRY_CAST(REPLACE(NULLIF(TRIM("Poverty Rate (5 Yr ACS)"), ''), '%', '') AS FLOAT) IS NULL)
   FROM PUBLIC.bronze.CENSUS_TRACT)                             AS poverty_rate_parse_fails,
  (SELECT COUNT_IF(NULLIF(TRIM("Percent Black (all ethnicities) (5 Yr ACS)"), '') IS NOT NULL
       AND TRY_CAST(REPLACE(NULLIF(TRIM("Percent Black (all ethnicities) (5 Yr ACS)"), ''), '%', '') AS FLOAT) IS NULL)
   FROM PUBLIC.bronze.CENSUS_TRACT)                             AS percent_black_parse_fails,
  (SELECT COUNT_IF(NULLIF(TRIM("Percent Hispanic (any race) (5 Yr ACS)"), '') IS NOT NULL
       AND TRY_CAST(REPLACE(NULLIF(TRIM("Percent Hispanic (any race) (5 Yr ACS)"), ''), '%', '') AS FLOAT) IS NULL)
   FROM PUBLIC.bronze.CENSUS_TRACT)                             AS percent_hispanic_parse_fails,
  (SELECT COUNT_IF(NULLIF(TRIM("Homeownership Rate (5 Yr ACS)"), '') IS NOT NULL
       AND TRY_CAST(REPLACE(NULLIF(TRIM("Homeownership Rate (5 Yr ACS)"), ''), '%', '') AS FLOAT) IS NULL)
   FROM PUBLIC.bronze.CENSUS_TRACT)                             AS homeownership_rate_parse_fails,
  (SELECT COUNT_IF(NULLIF(TRIM("Disability Rate (5 Yr ACS)"), '') IS NOT NULL
       AND TRY_CAST(REPLACE(NULLIF(TRIM("Disability Rate (5 Yr ACS)"), ''), '%', '') AS FLOAT) IS NULL)
   FROM PUBLIC.bronze.CENSUS_TRACT)                             AS disability_rate_parse_fails
;

## Step 4 - Confirm schema

Verify `CENSUS_TRACT_TRANSFORMED` has the expected types, the two *Food Bank 2* columns are gone, and the three decomposed `Geography` columns are present.

In [ ]:
%%sql -r dataframe_6
DESCRIBE TABLE PUBLIC.silver.CENSUS_TRACT;